In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset
from transformers import WhisperForAudioClassification, WhisperProcessor
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, recall_score, precision_score, f1_score, accuracy_score, classification_report, balanced_accuracy_score
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
import time
from tqdm import tqdm
import librosa
import random
from sklearn.model_selection import KFold

In [ ]:
# Defining a function to set or change the seed for random number generation at each step.

def reset_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available:
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

if torch.cuda.is_available():
  device = 'cuda'
else:
  device = 'cpu'

device

## data preprocessing

In [ ]:
reset_seed(42)

whisper_tiny_processor = WhisperProcessor.from_pretrained('w_tiny')


emo2num = {
    'H' : 0,
    'N' : 1,
    'S' : 2,
    'A' : 3,
    'W' : 4
}

num2emo = {
    0 : 'Happy',
    1 : 'Neutral',
    2 : 'Sad',
    3 : 'Anger',
    4 : 'Wonder'
}


audios = []
sample_rates = []
emotions = []
labels = []
file_names = []
file_paths = []
features = []

pattern = r'^...(.)'

for root, folders, files in os.walk('shemo'):
    for filename in files:
        file_path = os.path.join(root, filename)
        audio_data, sr = librosa.load(file_path, sr=16000)
        
        input_features = whisper_tiny_processor(audio_data, sampling_rate=sr, return_tensors="pt").input_features
    

        match = re.search(pattern, filename)
        if match:
            char = match.group(1)
            if char in ['H', 'N', 'S', 'W', 'A']:
                audios.append(audio_data)
                sample_rates.append(sr)
                # emotion = 'H' if char == 'W' else char
                emotion = char
                label = emo2num[emotion]

                emotions.append(emotion)
                labels.append(label)
                features.append(input_features.squeeze(0))
                file_names.append(filename)
                file_paths.append(file_path)

In [ ]:
kf = KFold(n_splits=5, random_state=42, shuffle=True)
kf.get_n_splits(features)

In [ ]:
for i, (train_index, test_index) in enumerate(kf.split(features)):
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}")
    print(f"  Test:  index={test_index}")

In [ ]:
(train_index1, test_index1), (train_index2, test_index2),(train_index3, test_index3),(train_index4, test_index4),(train_index5, test_index5) = kf.split(features)

In [ ]:
################# Train_features
trainF1features = [value for index, value in enumerate(features) if index in train_index1]
trainF2features = [value for index, value in enumerate(features) if index in train_index2]
trainF3features = [value for index, value in enumerate(features) if index in train_index3]
trainF4features = [value for index, value in enumerate(features) if index in train_index4]
trainF5features = [value for index, value in enumerate(features) if index in train_index5]

################# Train_labels
trainF1labels = [value for index, value in enumerate(labels) if index in train_index1]
trainF2labels = [value for index, value in enumerate(labels) if index in train_index2]
trainF3labels = [value for index, value in enumerate(labels) if index in train_index3]
trainF4labels = [value for index, value in enumerate(labels) if index in train_index4]
trainF5labels = [value for index, value in enumerate(labels) if index in train_index5]

################# Test_features
testF1features = [value for index, value in enumerate(features) if index in test_index1]
testF2features = [value for index, value in enumerate(features) if index in test_index2]
testF3features = [value for index, value in enumerate(features) if index in test_index3]
testF4features = [value for index, value in enumerate(features) if index in test_index4]
testF5features = [value for index, value in enumerate(features) if index in test_index5]

################# Test_labels
testF1labels = [value for index, value in enumerate(labels) if index in test_index1]
testF2labels = [value for index, value in enumerate(labels) if index in test_index2]
testF3labels = [value for index, value in enumerate(labels) if index in test_index3]
testF4labels = [value for index, value in enumerate(labels) if index in test_index4]
testF5labels = [value for index, value in enumerate(labels) if index in test_index5]

################## Train_names
trainF1names = [value for index, value in enumerate(file_names) if index in train_index1]
trainF2names = [value for index, value in enumerate(file_names) if index in train_index2]
trainF3names = [value for index, value in enumerate(file_names) if index in train_index3]
trainF4names = [value for index, value in enumerate(file_names) if index in train_index4]
trainF5names = [value for index, value in enumerate(file_names) if index in train_index5]

################# Train_paths
trainF1paths = [value for index, value in enumerate(file_paths) if index in train_index1]
trainF2paths = [value for index, value in enumerate(file_paths) if index in train_index2]
trainF3paths = [value for index, value in enumerate(file_paths) if index in train_index3]
trainF4paths = [value for index, value in enumerate(file_paths) if index in train_index4]
trainF5paths = [value for index, value in enumerate(file_paths) if index in train_index5]

################# Test_names
testF1names = [value for index, value in enumerate(file_names) if index in test_index1]
testF2names = [value for index, value in enumerate(file_names) if index in test_index2]
testF3names = [value for index, value in enumerate(file_names) if index in test_index3]
testF4names = [value for index, value in enumerate(file_names) if index in test_index4]
testF5names = [value for index, value in enumerate(file_names) if index in test_index5]

################# Test_paths
testF1paths = [value for index, value in enumerate(file_paths) if index in test_index1]
testF2paths = [value for index, value in enumerate(file_paths) if index in test_index2]
testF3paths = [value for index, value in enumerate(file_paths) if index in test_index3]
testF4paths = [value for index, value in enumerate(file_paths) if index in test_index4]
testF5paths = [value for index, value in enumerate(file_paths) if index in test_index5]

In [ ]:
reset_seed(42)
class AudioDataSet(Dataset):
    def __init__(self, features, labels, file_names=None, file_paths=None):
        self.features = features
        self.labels = labels
        self.index = list(range(len(labels)))
        self.filename = file_names
        self.filepath = file_paths


    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        # Assuming features[index] is a processed audio feature and labels[index] is the corresponding label
        sample = {
            'feature': torch.Tensor(self.features[index]),  # Convert to PyTorch tensor if not already
            'label': torch.Tensor([self.labels[index]]),  # Convert to PyTorch tensor if not already
            'name' : self.filename[index],
            'path' : self.filepath[index],
            'gender': self.filename[index][0],
            'index': self.index[index]
        }


        return sample


Fold1train = AudioDataSet(trainF1features, trainF1labels, trainF1names, trainF1paths)
Fold1test = AudioDataSet(testF1features, testF1labels, testF1names, testF1paths)

Fold2train = AudioDataSet(trainF2features, trainF2labels, trainF2names, trainF2paths)
Fold2test = AudioDataSet(testF2features, testF2labels, testF2names, testF2paths)

Fold3train = AudioDataSet(trainF3features, trainF3labels, trainF3names, trainF3paths)
Fold3test = AudioDataSet(testF3features, testF3labels, testF3names, testF3paths)

Fold4train = AudioDataSet(trainF4features, trainF4labels, trainF4names, trainF4paths)
Fold4test = AudioDataSet(testF4features, testF4labels, testF4names, testF4paths)

Fold5train = AudioDataSet(trainF5features, trainF5labels, trainF5names, trainF5paths)
Fold5test = AudioDataSet(testF5features, testF5labels, testF5names, testF5paths)

In [ ]:
folds_trains = [Fold1train, Fold2train, Fold3train, Fold4train, Fold5train]
folds_vals = [Fold1test, Fold2test, Fold3test, Fold4test, Fold5test]
folds_indices = [train_index1, train_index2, train_index3, train_index4, train_index5]

## train test and arch

In [ ]:
reset_seed(42)
def train(model, model_name, optimizer, loss_fn, train_loader, val_loader, epochs =40, patience=4):
  training_losses = []
  valid_losses = []
  durations = []
  best_val_loss = float('inf')
  best_val_accuracy = 0.0
  current_patience = 0

  for epoch in range(epochs):
    training_loss = []
    valid_loss = []
    start_time = time.time()

    model.train()
    for batch in tqdm(train_loader, desc=f'Epoch {epoch}/{epochs}', leave=False): # Each batch, One Iteration
      optimizer.zero_grad()
      inputs, targets = batch['feature'], batch['label'].squeeze(-1).long()
      inputs, targets = inputs.to(device), targets.to(device)
      output = model(inputs)
      loss = loss_fn(output, targets)
      loss.backward()
      optimizer.step()
      inputs, targets, output = inputs.to('cpu'), targets.to('cpu'), output.to('cpu') # Return tensors to cpu after each iteration

      training_loss.append(loss.data.item())
    training_loss = np.mean(training_loss)
    training_losses.append(training_loss)
    

    model.eval()
    num_correct = 0
    num_examples = 0
    for batch in tqdm(val_loader, desc=f'Epoch {epoch}/{epochs}', leave=False):
      inputs, targets = batch['feature'], batch['label'].squeeze(-1).long()
      inputs, targets = inputs.to(device), targets.to(device)
      output = model(inputs)
      loss = loss_fn(output, targets)
      valid_loss.append(loss.data.item())
      correct = torch.eq(torch.argmax(F.softmax(output, dim=1), dim=1), targets.view(-1))
      num_correct += torch.sum(correct).item()
      num_examples += correct.shape[0]
      inputs, targets, output = inputs.to('cpu'), targets.to('cpu'), output.to('cpu') # Return tensors to cpu after each iteration
      
    valid_loss = np.mean(valid_loss)
    valid_losses.append(valid_loss)
    end_time = time.time()
    duration = end_time - start_time
    durations.append(duration)
    valid_accuracy = num_correct / num_examples

    print(f'Epoch: {epoch+1} finished in {duration:.2f} seconds, Training Loss: {training_loss: .2f}, Validation Loss: {valid_loss:.2f}, Validation Accuracy: {valid_accuracy : .2f}')

    # Check for improvement
    if valid_loss < best_val_loss:
        best_val_loss = valid_loss
        current_patience = 0
    else:
        current_patience += 1

    if valid_accuracy > best_val_accuracy:
      best_val_accuracy = valid_accuracy
      torch.save(model.state_dict(), f'{model_name}.bin') # Saving the best model because I want to use the best checkpoints of the model to test it

    # Early stopping logic
    if current_patience >= patience:
        print(f"Early stopping at epoch {epoch+1}. Because Validation loss increased for {patience} consecutive epochs.")
        break

  print(f"Each epoch took {np.mean(durations):.2f} seconds in average.")

  return training_losses, valid_losses

In [ ]:
def test(model, loss_fn, test_loader):
  model.eval()
  test_loss = []
  labels = []
  preds = []
  for batch in test_loader:
    inputs, targets = batch['feature'], batch['label'].squeeze(-1).long()

    for target in targets:
      labels.append(target.item())
    inputs, targets = inputs.to(device), targets.to(device)
    output = model(inputs)
    for out in output:
      preds.append(torch.argmax(F.softmax(out, dim=0), dim=0).item())
    loss = loss_fn(output, targets)
    test_loss.append(loss.data.item())
    inputs, targets, output = inputs.to('cpu'), targets.to('cpu'), output.to('cpu') # Return tensors to cpu after each iteration


  #######################################################################################################
  accuracy_per_class = {}
  unique_classes = set(labels)
  for class_label in unique_classes:
      indices = [i for i, y in enumerate(labels) if y == class_label]
      class_accuracy = accuracy_score([labels[i] for i in indices], [preds[i] for i in indices])
      accuracy_per_class[class_label] = class_accuracy

  w_acc = sum(accuracy_per_class.values())/len(accuracy_per_class.values())
  #######################################################################################################


  test_loss = np.mean(test_loss)
  w_acc = accuracy_score(labels, preds)
  u_acc = sum(accuracy_per_class.values())/len(accuracy_per_class.values())
  precision = precision_score(labels, preds, average='weighted')
  recall = recall_score(labels, preds, average='weighted')
  f1 = f1_score(labels, preds, average='weighted')
  cm = confusion_matrix(labels, preds)
  sns.set(font_scale=1.2)  # Adjust the font scale for better readability
  sns.heatmap(cm, annot=True, fmt='g', cmap='Greens', 
              xticklabels=['Happy', 'Neutral', 'Sad', 'Anger', 'Wonder'], 
              yticklabels=['Happy', 'Neutral', 'Sad', 'Anger', 'Wonder'])
  plt.xlabel('Predicted')
  plt.ylabel('Actual')
  plt.show()

  print(f'Testing finished with Loss: {test_loss: .2f} | U_Accuracy: {u_acc : .2f} | W_Accuracy: {w_acc : .2f} | Precision: {precision : .2f} | Recall: {recall : .2f} | f1: {f1 : .2f}')

  print("Accuracy per class:")
  for class_label, accuracy in accuracy_per_class.items():
      print(f"Class {class_label}: {accuracy}")

In [ ]:
reset_seed(42)

class WhisAttSumModel (nn.Module):
  def __init__(self):
    super(WhisAttSumModel, self).__init__()
    self.whisperencoder = WhisperForAudioClassification.from_pretrained('whisper_small').encoder.to(device)
    self.projector = WhisperForAudioClassification.from_pretrained('whisper_small').projector.to(device)
    self.layernorm = nn.LayerNorm(256)
    self.attention_scorer = nn.Linear(in_features=256, out_features=1, bias=True)
    self.classifier = torch.nn.Linear(in_features=256, out_features=5)

  def forward(self, x):
    out = self.whisperencoder(x).last_hidden_state
    out = self.projector(out)
    out = self.layernorm(out)
    attention_scores = self.attention_scorer(out)
    attention_weights = F.softmax(attention_scores, dim=1)
    weighted_output = out*attention_weights
    weighted_mean_pooled_output = weighted_output.sum(dim=1)
    out = self.classifier(weighted_mean_pooled_output)

    return out

## training

In [ ]:
reset_seed(42)
counter = 1
loader_lists = []

for trainset_indices, trainset, valset in zip(folds_indices, folds_trains, folds_vals):
    audios_to_aug = [audio for (ind,audio) in enumerate(audios) if ind in trainset_indices]
    audio_gender_to_aug = [name[0] for (ind,name) in enumerate(file_names) if ind in trainset_indices]
    audio_emo_to_aug = [emotion for (ind,emotion) in enumerate(emotions) if ind in trainset_indices]

    segments ={}


    for emotion in ['H', 'W', 'S', 'A', 'N']:

        audiosF1 = [audio_data[:len(audio_data)//2] for (audio_data, gen, emo) in zip(audios_to_aug, audio_gender_to_aug, audio_emo_to_aug) if (gen == 'F' and emo == emotion)]
        audiosF2 = [audio_data[len(audio_data)//2:] for (audio_data, gen, emo) in zip(audios_to_aug, audio_gender_to_aug, audio_emo_to_aug) if (gen == 'F' and emo == emotion)]
        audiosM1 = [audio_data[:len(audio_data)//2] for (audio_data, gen, emo) in zip(audios_to_aug, audio_gender_to_aug, audio_emo_to_aug) if (gen == 'M' and emo == emotion)]
        audiosM2 = [audio_data[len(audio_data)//2:] for (audio_data, gen, emo) in zip(audios_to_aug, audio_gender_to_aug, audio_emo_to_aug) if (gen == 'M' and emo == emotion)]

        segments[emotion] = [(audiosF1, audiosF2), (audiosM1, audiosM2)]

    
    augmented_audios = []
    augmented_labels = []
    augmented_emotions = []

    augmentation_need = {
        'H': 9,
        'W': 9,
        'S': 4,
        'A': 1,
        'N': 1
        }
    
    for category in segments:
        for gender_segments in segments[category]:
            for audio in gender_segments[0]:
                choice = random.sample(gender_segments[1], augmentation_need[category])
                for ch in choice:
                    new_speech = np.concatenate((audio, ch))
                    augmented_audios.append(new_speech)
                    augmented_emotions.append(category)
                    augmented_labels.append(emo2num[category])


    augmented_samplerates = []
    augmeted_features = []


    for audio in augmented_audios:

        input_features = whisper_tiny_processor(audio, sampling_rate=16000, return_tensors="pt").input_features

        augmented_samplerates.append(16000)
        augmeted_features.append(input_features.squeeze(0))


    Aug = AudioDataSet(augmeted_features, augmented_labels, [' ']*len(augmeted_features), [' ']*len(augmeted_features))
    concatenated_dataset = ConcatDataset([trainset, Aug])


    train_loader = DataLoader(concatenated_dataset, batch_size=4, shuffle=True)
    val_loader = DataLoader(valset, batch_size=4, shuffle=False)


    model = WhisAttSumModel()

    for param in model.parameters():
        param.requires_grad = False

    for param in model.projector.parameters():
        param.requires_grad = True

    for param in model.attention_scorer.parameters():
        param.requires_grad = True

    for param in model.classifier.parameters():
        param.requeres_grad = True

    for param in model.layernorm.parameters():
        param.requires_grad = True

    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    training_losses1, valid_losses1 = train(model, f'whisper_tiny_aug_fold{counter}', optimizer, torch.nn.CrossEntropyLoss(), train_loader, val_loader, patience=40)
    

    plt.plot(training_losses1, label='Training Loss')
    plt.plot(valid_losses1, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

    model.load_state_dict(torch.load(f'whisper_tiny_aug_fold{counter}.bin'))
    test(model, torch.nn.CrossEntropyLoss(), val_loader)
    counter += 1